In [3]:
!pip install pandas-ta --no-cache-dir --quiet
!pip install sklearn xgboost --no-cache-dir --quiet
!pip install xgboost --no-chache-dir --quiet
try:
    from xgboost import XGBClassifier
    import pandas as pd
    import numpy as np
    import pandas_ta as ta
    import yfinance as yf
    import matplotlib.pyplot as plt
    from xgboost import XGBClassifier
    from sklearn.metrics import accuracy_score, classification_report

    import warnings
    warnings.filterwarnings('ignore')

    np.random.seed(42)
    print("✅ All libraries sucessfully imported!")
except ImportError as e:
    print(f"❌ Error: {e}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 41.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 239.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 316.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 306.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 287.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 245.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 303.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run 

In [4]:
NVDA = yf.Ticker('NVDA')
df = NVDA.history(period='max')

df['Next_Close'] = df['Close'].shift(-1)
df['Return'] = df['Close'].pct_change() #.shift(-1) --- No Data Leakage ---> R_t = (P_t - P_{t-1}) / P_t
df['Label'] = np.where(df['Next_Close'] > df['Close'], 1, 0) # 1: UP, 0: DOWN

horizons = [1, 2, 5, 21, 55, 100, 200]

time_indicators = []

for h in horizons:
    ratio_col = f"Close_Ratio_{h}"
    trend_col = f"Trend_{h}"
    df[ratio_col] = df['Close'] / df['Close'].rolling(h).mean()
    df[trend_col] = df.shift(1)['Label'].rolling(h).sum()
    time_indicators += [ratio_col, trend_col]

df.ta.rsi(length=14, append=True)
df.ta.bbands(length=20, std=2, append=True)
df.ta.macd(fast=12, slow=26, signal=9, append=True)

df.dropna(inplace=True)

In [5]:
features = ['Volume'] + time_indicators + [col for col in df.columns if col.startswith(('SMA','EMA','MACD', 'RSI', 'BB'))]

X = df[features].to_numpy()
y = df['Label'].to_numpy()

split_idx = len(df) - 500

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

In [6]:
model = XGBClassifier(
    eval_metric='logloss',
    n_estimators=50,
    max_depth=3,
    learning_rate=0.1,

    # Reg Pars
    gamma=1.0,
    subsample=0.7,         # Use subsampe_% of data to build trees
    reg_alpha=0.1,         # L1 (Lasso) Reg.
    reg_lambda=1.0,        # L2 (Ridge) Reg.
    #colsample_bytree=0.7, # Use colsample_bytree_% of features

    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train, verbose=True)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=1.0,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=50, n_jobs=-1,
              num_parallel_tree=None, ...)

In [12]:
IS_pred = model.predict(X_train)
OOS_preds = model.predict(X_test)

IS_accuracy = accuracy_score(y_train, IS_pred)
OOS_accuracy = accuracy_score(y_test, OOS_preds)

print("-" * 24 + " XGBoostClassifier Analysis "+  "-" * 23)

print(f"")
print(f" Out-Of-Sample Accuracy: {OOS_accuracy * 100:.2f}%")
print(f" In-Sample Accuracy: {IS_accuracy * 100:.2f}%")
print(f"")

# target_names = ['UP', 'DOWN'] o ['DOWN','UP']?
# print(model.classes_) # >>> [0 1] => ['DOWN','UP']

print("-" * 26 + " CLASSIFICATION REPORT "+  "-" * 26)
print(f"")
print(classification_report(y_test, OOS_preds, target_names=['DOWN', 'UP']))

------------------------ XGBoostClassifier Analysis -----------------------

 Out-Of-Sample Accuracy: 53.60%
 In-Sample Accuracy: 63.04%

-------------------------- CLASSIFICATION REPORT --------------------------

              precision    recall  f1-score   support

        DOWN       0.51      0.27      0.35       235
          UP       0.54      0.77      0.64       265

    accuracy                           0.54       500
   macro avg       0.53      0.52      0.50       500
weighted avg       0.53      0.54      0.50       500



In [11]:
# predict_proba
OOS_probabilities = model.predict_proba(X_test)
next_test_days=20
date_test = df.index[-next_test_days:]
next_test_days_probabilities = OOS_probabilities[-next_test_days:]
no_trade_conf_level = 55 # %

df_prob = pd.DataFrame(
    next_test_days_probabilities,
    columns=['Prob_DOWN', 'Prob_UP']
)

df_prob['Prob_DOWN'] = df_prob['Prob_DOWN'].map(lambda x: f"{x*100:.1f}%")
df_prob['Prob_UP'] = df_prob['Prob_UP'].map(lambda x: f"{x*100:.1f}%")

df_prob.index = [f"Day t+{i+1}" for i in range(next_test_days)]

print(f" Predicted UP-DOWN probabilities for the next {next_test_days} days:\n")
print("-" * 75)

correct_preds = 0
no_trade_days = 0
preds_num = next_test_days

for idx, (date, prob) in enumerate(zip(date_test, next_test_days_probabilities)):
    p_down, p_up = prob[0], prob[1]

    if max(p_down, p_up) * 100 < no_trade_conf_level:
        preds_num -= 1
        no_trade_days += 1
        continue

    label_mapping = {1: "🟢 UP", 0: "🔴 DOWN"}
    actual_label_val = df['Label'].iloc[split_idx + idx]
    actual_return = label_mapping[actual_label_val]

    if p_up > p_down:
        prediction = "🟢 UP  "
        if actual_label_val == 1:
            correct_preds += 1
        confidence = p_up * 100
        print(f"Date: {date.strftime('%Y-%m-%d')} | {prediction} (Confidence: {confidence:.1f}%) | [DOWN: {p_down*100:.1f}% | Actual Return {actual_return}")
    else:
        prediction = "🔴 DOWN"
        if actual_label_val == 0:
            correct_preds += 1
        confidence = p_down * 100
        print(f"Date: {date.strftime('%Y-%m-%d')} | {prediction} (Confidence: {confidence:.1f}%) | [UP:  {p_up*100:.1f}%] | Actual Return {actual_return}")

print("-" * 75)
print(f"- Date range: {df.index[0].strftime('%Y-%m-%d')} - {df.index[-1].strftime('%Y-%m-%d')} - Total: {next_test_days}")
print(f"- Number of predictions: {preds_num}")
print(f"- Number of no trade days: {no_trade_days}")
print(f"- Correct predictions: {correct_preds}")
print(f"- Accuracy: {correct_preds / preds_num * 100:.2f}%")
print("-" * 75)

 Predicted UP-DOWN probabilities for the next 20 days:

---------------------------------------------------------------------------
Date: 2026-05-18 | 🟢 UP   (Confidence: 63.8%) | [DOWN: 36.2% | Actual Return 🟢 UP
Date: 2026-05-19 | 🟢 UP   (Confidence: 66.3%) | [DOWN: 33.7% | Actual Return 🔴 DOWN
Date: 2026-05-20 | 🟢 UP   (Confidence: 65.9%) | [DOWN: 34.1% | Actual Return 🔴 DOWN
Date: 2026-05-28 | 🔴 DOWN (Confidence: 57.0%) | [UP:  43.0%] | Actual Return 🔴 DOWN
Date: 2026-06-03 | 🔴 DOWN (Confidence: 59.1%) | [UP:  40.9%] | Actual Return 🔴 DOWN
Date: 2026-06-08 | 🔴 DOWN (Confidence: 56.0%) | [UP:  44.0%] | Actual Return 🟢 UP
---------------------------------------------------------------------------
- Date range: 1999-11-05 - 2026-06-15 - Total: 20
- Number of predictions: 6
- Number of no trade days: 14
- Correct predictions: 3
- Accuracy: 50.00%
---------------------------------------------------------------------------
